# 🔍 Chapter 3: Unsupervised Learning and Preprocessing
**Reference:** *Introduction to Machine Learning with Python* by Andreas C. Müller & Sarah Guido

---

## 3.1 Introduction to Unsupervised Learning
Unlike supervised learning, unsupervised learning algorithms extract patterns and structure from data without any known output labels ($y$). The book divides unsupervised learning into two main categories:

1. **Unsupervised Transformations:** Algorithms that create a new representation of the data. This is often used for dimensionality reduction (compressing the data into essential features for visualization or to reduce computation time) or for finding "parts" that make up the data.
2. **Clustering Algorithms:** Algorithms that partition data into distinct groups of similar items.

A major challenge in unsupervised learning is evaluating whether the algorithm learned something useful, because we don't have ground-truth labels to calculate accuracy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_digits, make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from scipy.cluster.hierarchy import dendrogram, ward

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
np.random.seed(42)

print("Libraries successfully imported for Chapter 3.")

## 3.2 Preprocessing and Scaling
Before applying unsupervised (or supervised) algorithms, data must often be rescaled. Algorithms like SVMs, Neural Networks, and distance-based clustering (like K-Means) are highly sensitive to the scale of the data.

Scikit-learn provides several scalers:
- **StandardScaler:** Ensures each feature has a mean of 0 and variance of 1. Does not guarantee exact minimum and maximum values.
- **MinMaxScaler:** Shifts the data such that all features are exactly between 0 and 1.
- **RobustScaler:** Uses the median and quartiles, making it ignore extreme outliers.
- **Normalizer:** Scales each data point such that the feature vector has a Euclidean length of 1.

**CRITICAL RULE:** You must fit the scaler on the *Training Data* only, and then use `.transform()` on both the Training and Test sets. Never `.fit()` on the Test set.

In [ ]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, random_state=1)

# Instantiate and fit the MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(X_train)

# Transform both train and test data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Original X_train min: {X_train.min(axis=0)[0]:.3f}, max: {X_train.max(axis=0)[0]:.3f}")
print(f"Scaled X_train min  : {X_train_scaled.min(axis=0)[0]:.3f}, max: {X_train_scaled.max(axis=0)[0]:.3f}")
print(f"Scaled X_test min   : {X_test_scaled.min(axis=0)[0]:.3f}, max: {X_test_scaled.max(axis=0)[0]:.3f}")
print("Notice that X_test_scaled min/max might not be exactly 0 and 1. This is correct and prevents data leakage.")

## 3.3 Dimensionality Reduction: Principal Component Analysis (PCA)
PCA is a method that rotates the dataset in a way such that the rotated features are statistically uncorrelated. It identifies the directions (principal components) that contain the most variance in the data.

By keeping only the first few principal components, we can reduce data with 30+ features down to 2 features, allowing us to visualize complex relationships on a 2D scatter plot.

In [ ]:
# It is highly recommended to use StandardScaler before applying PCA
st_scaler = StandardScaler()
st_scaler.fit(cancer.data)
X_scaled = st_scaler.transform(cancer.data)

# Keep the first two principal components of the data
pca = PCA(n_components=2)
pca.fit(X_scaled)

# Transform the data onto the first two principal components
X_pca = pca.transform(X_scaled)

print(f"Original shape: {X_scaled.shape}")
print(f"Reduced shape : {X_pca.shape}")

# Plot the first two principal components
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[cancer.target == 0, 0], X_pca[cancer.target == 0, 1], 
            color='red', alpha=0.5, label=cancer.target_names[0])
plt.scatter(X_pca[cancer.target == 1, 0], X_pca[cancer.target == 1, 1], 
            color='blue', alpha=0.5, label=cancer.target_names[1])

plt.xlabel("First Principal Component")
plt.ylabel("Second Principal Component")
plt.title("PCA of Breast Cancer Dataset")
plt.legend()
plt.show()
print("PCA allows us to see that the two classes are remarkably well separated in a 2D space, despite the original data having 30 features.")

## 3.4 Manifold Learning with t-SNE
While PCA is great for finding linear transformations, it often fails to uncover complex, non-linear relationships. **Manifold learning** algorithms like **t-SNE** (t-Distributed Stochastic Neighbor Embedding) are designed specifically to visualize high-dimensional data.

t-SNE works by trying to preserve the distances between data points. If two points are close in the high-dimensional space, t-SNE tries to keep them close in the 2D space. It is incredibly effective for visualizing clusters, but it is **not** used for transforming new test data (it has no `.transform()` method, only `.fit_transform()`).

In [ ]:
# Load the Digits dataset (8x8 images of handwritten digits)
digits = load_digits()

# Apply t-SNE
tsne = TSNE(random_state=42)
digits_tsne = tsne.fit_transform(digits.data)

plt.figure(figsize=(10, 8))
colors = ["#476A2A", "#7851B8", "#BD3430", "#4A2D4E", "#875525",
          "#A83683", "#4E655E", "#853541", "#3A3120", "#535D8E"]

for i in range(len(digits.data)):
    # Plot each point as its actual digit class (0-9) instead of a dot
    plt.text(digits_tsne[i, 0], digits_tsne[i, 1], str(digits.target[i]),
             color=colors[digits.target[i]], fontdict={'weight': 'bold', 'size': 9})

plt.xlim(digits_tsne[:, 0].min(), digits_tsne[:, 0].max() + 1)
plt.ylim(digits_tsne[:, 1].min(), digits_tsne[:, 1].max() + 1)
plt.title("t-SNE Visualization of the Digits Dataset")
plt.xlabel("t-SNE feature 0")
plt.ylabel("t-SNE feature 1")
plt.show()
print("Notice how t-SNE perfectly separates the different handwritten digits into distinct, non-overlapping clusters without knowing their true labels!")

## 3.5 Clustering: k-Means
k-Means is the simplest and most commonly used clustering algorithm. It tries to find cluster centers (centroids) that are representative of certain regions of the data.

**How it works:**
1. Pick $k$ random cluster centers.
2. Assign each data point to the closest cluster center.
3. Move the cluster centers to the mean of all points assigned to it.
4. Repeat until the cluster centers stop moving.

**Drawbacks:** k-Means assumes clusters are convex (spherical) and have similar sizes. It fails completely on complex, elongated shapes.

In [ ]:
# Generate synthetic blob data
X_blobs, y_blobs = make_blobs(n_samples=300, random_state=1)

# Apply k-Means
kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)
kmeans.fit(X_blobs)

plt.figure(figsize=(8, 6))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans.labels_, cmap='viridis', s=50, alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            marker='^', s=200, c='red', label='Centroids')
plt.title("k-Means Clustering (k=3)")
plt.legend()
plt.show()

## 3.6 Agglomerative (Hierarchical) Clustering
Agglomerative clustering is a "bottom-up" approach. 
It starts by declaring each data point as its own cluster. Then, it searches for the two most similar clusters and merges them. It repeats this process until all points are merged into a single giant cluster.

Unlike k-Means, hierarchical clustering allows us to visualize the entire history of cluster merges using a tree-like plot called a **Dendrogram**.

In [ ]:
# Generate small synthetic data for readable dendrogram
X_small, _ = make_blobs(n_samples=30, random_state=0)

# Apply the Ward linkage function to compute the cluster distances
linkage_array = ward(X_small)

plt.figure(figsize=(10, 6))
# Plot the dendrogram
dendrogram(linkage_array)

# Mark the cuts in the tree that signify two or three clusters
ax = plt.gca()
bounds = ax.get_xbound()
ax.plot(bounds, [15, 15], '--', c='k')
ax.plot(bounds, [7, 7], '--', c='k')
plt.text(bounds[1], 15, ' 2 clusters', va='center', fontdict={'size': 12})
plt.text(bounds[1], 7, ' 3 clusters', va='center', fontdict={'size': 12})
plt.title("Dendrogram for Agglomerative Clustering")
plt.xlabel("Sample index")
plt.ylabel("Cluster distance")
plt.show()

## 3.7 DBSCAN (Density-Based Spatial Clustering)
DBSCAN is an incredibly powerful clustering algorithm because it does not require you to set the number of clusters ($k$) ahead of time, and it can capture clusters of very complex shapes.

It works by identifying "dense" regions of data. If there are many points packed closely together, they form a cluster. Points that are in low-density regions (far away from everything else) are categorized as **noise** (outliers).

Let's test k-Means and DBSCAN on a "Two Moons" dataset, which is a classic non-spherical shape.

In [ ]:
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=0)

# Scale the data (DBSCAN relies heavily on distance thresholds)
scaler = StandardScaler()
scaler.fit(X_moons)
X_scaled_moons = scaler.transform(X_moons)

# Apply K-Means (will fail)
kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
kmeans_clusters = kmeans.fit_predict(X_scaled_moons)

# Apply DBSCAN (will succeed)
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_clusters = dbscan.fit_predict(X_scaled_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_scaled_moons[:, 0], X_scaled_moons[:, 1], c=kmeans_clusters, cmap='viridis', s=60, alpha=0.8)
axes[0].set_title("k-Means (Fails on complex shapes)")

axes[1].scatter(X_scaled_moons[:, 0], X_scaled_moons[:, 1], c=dbscan_clusters, cmap='viridis', s=60, alpha=0.8)
axes[1].set_title("DBSCAN (Succeeds based on density)")

plt.show()
print("DBSCAN effortlessly identifies the two distinct moon shapes because it groups continuous dense regions, whereas k-Means foolishly tries to slice the space in half with a straight line.")

## 3.8 Chapter Summary
Key takeaways from Chapter 3:
1. **Scaling is crucial:** Unsupervised algorithms (PCA, K-Means, DBSCAN) calculate distances. If you don't scale features using `StandardScaler` or `MinMaxScaler`, features with larger numbers will unfairly dominate the algorithm.
2. **PCA is for compression and visualization:** It finds linear orthogonal axes of maximum variance.
3. **t-SNE is for visualization only:** It beautifully unwraps complex high-dimensional manifolds into 2D, but cannot be applied to new, unseen test data.
4. **No single clustering algorithm is perfect:**
   - *k-Means:* Fast, scalable, but assumes spherical clusters and requires guessing $k$.
   - *Agglomerative:* Creates interpretable dendrograms but is slow.
   - *DBSCAN:* Detects outliers and complex shapes, doesn't require $k$, but requires careful tuning of the `eps` (radius) parameter.